# Contrastive video textures — Colab demo

This notebook clones the project, installs dependencies with **uv**, downloads the checkpoints the code expects on fixed paths (SlowFast, VGGish, **SuperSloMo**), and runs a sample **training** command.

For synthesis / evaluation (`--evaluate`), you must train or resume checkpoints first; `--SF` only controls frame interpolation at evaluation time (see `contrastive_video_textures/main.py`), not SlowFast pretraining.

At the end you will find **FVD / Diversity Score**: how to call `evaluation.metrics` once you have real vs. generated **feature** arrays (the notebook does not download I3D; it demonstrates the API on synthetic features).

In [1]:
!git clone --quiet https://github.com/KoniHD/Berkeley-CS289A-Final-Project.git
%cd Berkeley-CS289A-Final-Project

/content/Berkeley-CS289A-Final-Project


*Note:* This part takes a while due to a lot of old legacy dependencies

In [2]:
# Install this package and its dependencies from pyproject.toml (not a requirements file).
!uv pip install --system -r pyproject.toml

Using Python 3.12.13 environment at: /usr
Resolved 122 packages in 3.95s
Prepared 36 packages in 25.33s
Uninstalled 22 packages in 709ms
Installed 36 packages in 194ms
Prepared 1 package without build isolation in 3m 47s
Installed 1 package in 7ms
 + av==17.0.1
 + detectron2==0.6 (from git+https://github.com/facebookresearch/detectron2.git@e0ec4e189d438848521aee7926f9900e114229f5)
 + fairscale==0.4.13
 + fvcore==0.1.5.post20221221
 + hydra-core==1.2.0
 - imageio-ffmpeg==0.6.0
 + imageio-ffmpeg==0.4.9
 + iopath==0.1.10
 + ipdb==0.13.13
 + jedi==0.20.0
 - numpy==2.0.2
 + numpy==1.26.4
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.1.3.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.1.105
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.1.105
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.1.105
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==8.9.2.26
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.0.2.54
 -

# Download pre-trained weights

## Download pre-trained ResNet, Slowfast, VGGish

*Note:* This part takes a while

In [3]:
import os
import torch
import shutil
from pathlib import Path

# 1. Mock the hardcoded directories for R3D and SlowFast
PRETRAINED = "/home/medhini/audio_video_gan/contrastive_video_textures/pretrained"
SF_CFG = "/home/medhini/audio_video_gan/contrastive_video_textures/slowfast_configs"
os.makedirs(PRETRAINED, exist_ok=True)
os.makedirs(SF_CFG, exist_ok=True)

# 2. Fast wgets for SlowFast into the hardcoded paths
print("1/3: Pulling SlowFast config & weights directly...")
!wget -qO {SF_CFG}/SLOWFAST_8X8_R50.yaml https://raw.githubusercontent.com/facebookresearch/SlowFast/master/configs/Kinetics/SLOWFAST_8x8_R50.yaml
!wget -qO {PRETRAINED}/SLOWFAST_8x8_R50.pkl https://dl.fbaipublicfiles.com/pyslowfast/model_zoo/kinetics400/SLOWFAST_8x8_R50.pkl

# 3. VGGish Surgery - SAVED LOCALLY THIS TIME
print("2/3: Pulling VGGish and running key surgery...")
!wget -qO vgg_raw.pth https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth
fixed_vgg = {k.replace("embeddings", "fc") if k.startswith("embeddings") else k: v for k, v in torch.load("vgg_raw.pth", map_location="cpu").items()}
# THE FIX: Save it locally exactly where main.py line 338 expects it!
torch.save(fixed_vgg, "pytorch_vggish.pth")

# 4. The R3D-18 Reality Check
print("3/3: Fetching custom 1039-class R3D-18 weights via gdown...")
!pip install -q gdown
!mkdir -p /tmp/r3d_pretrained
!gdown --folder https://drive.google.com/drive/folders/1xbYbZ7rpyjftI_KCk6YuL-XrfQDz7Yd4 -O /tmp/r3d_pretrained --remaining-ok

# Forcefully yank the file out of the GDrive folder structure into Medhini's path
print("Yanking r3d18_KM_200ep.pth into the hardcoded path...")
r3d_candidates = list(Path("/tmp/r3d_pretrained").rglob("r3d18_KM_200ep.pth"))
shutil.copy(r3d_candidates[0], f"{PRETRAINED}/r3d18_KM_200ep.pth")

print("Dirty prep complete! VGGish is local, R3D is hardcoded. You are good to go.")

1/3: Pulling SlowFast config & weights directly...
2/3: Pulling VGGish and running key surgery...
3/3: Fetching custom 1039-class R3D-18 weights via gdown...
Retrieving folder contents
Processing file 1U3hJGSchb8fNmlKe-oAvO5F2mP4Kde4X r2p1d18_K_200ep.pth
Processing file 17tHPKAaulga09P-5ZOKX44BdN-GmwBjl r2p1d34_K_200ep.pth
Processing file 11efqPBt6LAGJZUGhiDpc7ZX4HzuGmOJR r2p1d50_K_200ep.pth
Processing file 1D9VpovdlqlVAXyKyMQgLGx-uh5taazZo r2p1d50_KM_200ep.pth
Processing file 1Nb4abvIkkp_ydPFA9sNPT1WakoVKA8Fa r3d18_K_200ep.pth
Processing file 12FxrQY2hX-bINbmSrN9q2Z5zJguJhy6C r3d18_KM_200ep.pth
Processing file 1fFN5J2He6eTqMPRl_M9gFtFfpUmhtQc9 r3d34_K_200ep.pth
Processing file 1K9oiny9ENYODFxjFBTKdGeoDvOzPu4qQ r3d34_KM_200ep.pth
Processing file 1H52vT1T0sl7iWA7Up8wu1rSMFzgdwGZG r3d50_K_200ep.pth
Processing file 1fCKSlakRJ54b3pEWqgBmuJi0nF7HXQc0 r3d50_KM_200ep.pth
Processing file 1Z1agO6kKkMr-RcQz3DTptOORrqma1dQd r3d50_KMS_200ep.pth
Processing file 1aYlLkFu7uQr9sTKrkBlzPuGLgXK4M1p0 r3d

## Download SuperSloMo weights

In [4]:
# Run from repo root (after %cd Berkeley-CS289A-Final-Project)
import os, subprocess

ckpt_dir = "contrastive_video_textures/ckpt"
os.makedirs(ckpt_dir, exist_ok=True)
out = os.path.join(ckpt_dir, "SuperSloMo.ckpt")

subprocess.run(["pip", "install", "-q", "gdown"], check=True)
subprocess.run([
    "gdown", "1IvobLDbRiBgZr3ryCRrWL8xDbMZ-KnpF", "-O", out
], check=True)

import torch
d = torch.load(out, map_location="cpu")
assert "state_dictAT" in d and "state_dictFC" in d, f"Unexpected keys: {d.keys()}"
print("OK:", out)

OK: contrastive_video_textures/ckpt/SuperSloMo.ckpt


# Train encoder

In [ ]:
%cd contrastive_video_textures

In [ ]:
!python main.py \
    --vdata ../data \
    --model_type 1 \
    --window 20 \
    --stride 4 \
    --temp 0.1 \
    -th 0.0 \
    --batch_size 8 \
    -negs 14 \
    --video_list Clown-Fish \
    --enc_arch resnet18 \
    --lr 1e-4

# Generate video

In [ ]:
!python main.py \
    --vdata ../data \
    --model_type 1 \
    --window 20 \
    --stride 4 \
    --temp 0.1 \
    --threshold 0.3 \
    --batch_size 8 \
    --n_negs 14 \
    --video_list Clown-Fish \
    --enc_arch resnet18 \
    --evaluate \
    --overlay_png ../data/Clownfish.png

## FVD (Fréchet Video Distance)

In [8]:
!pwd

/content/Berkeley-CS289A-Final-Project/contrastive_video_textures


In [13]:
!python -m evaluation.extract_features \
  --real_glob ../data/*.mp4 ../results/Clown-Fish_original.mp4 \
  --fake_glob ../results/video_Clown-Fish_SF_5.mp4 \
  --out_dir ../results/fvd_features \
  --encoder resnet18

/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_video.py:6: UserWarning: The 'torchvision.transforms._functional_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms.functional' module instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_transforms_video.py:22: UserWarning: The 'torchvision.transforms._transforms_video' module is deprecated since 0.12 and will be removed in the future. Please use the 'torchvision.transforms' module instead.
  warnings.warn(
Loading encoder=resnet18 on cuda (frozen ImageNet/Kinetics weights, not your texture checkpoint) ...
Downloading: "https://download.pytorch.org/models/resnet18-5c106cde.pth" to ./pretrained/resnet18-5c106cde.pth
Real videos (2):
  encoding ../data/vtfishtk.mp4 ...
  encoding ../results/Clown-Fish_original.mp4 ...
Fake videos (1):
  encoding ../results/video_Clown-Fish_SF_5.mp4 ...
Saved ../results/fvd_features/

In [14]:
import os
from pathlib import Path

if Path("evaluation").is_dir():
    pass
elif Path("contrastive_video_textures/evaluation").is_dir():
    os.chdir("contrastive_video_textures")

import numpy as np
from evaluation.metrics import diversity_score, frechet_video_distance

feat_dir = Path("../results/fvd_features")
real_features = np.load(feat_dir / "real_features.npy")
fake_features = np.load(feat_dir / "fake_features.npy")

print("real_features", real_features.shape)
print("fake_features", fake_features.shape)
print("FVD:", frechet_video_distance(real_features, fake_features))

real_trans = np.load(feat_dir / "real_transitions.npy")
fake_trans = np.load(feat_dir / "fake_transitions.npy")
if real_trans.size:
    print("Diversity (real transitions):", diversity_score(real_trans))
if fake_trans.size:
    print("Diversity (fake transitions):", diversity_score(fake_trans))

real_features (79, 512)
fake_features (49, 512)
FVD: 321.03485274969216
Diversity (real transitions): 4.03622275827567
Diversity (fake transitions): 4.762986663721176


In [15]:
import numpy as np
from evaluation.metrics import diversity_score, frechet_video_distance

feat_dir = Path("../results/fvd_features")
real_features = np.load(feat_dir / "real_features.npy")
fake_features = np.load(feat_dir / "fake_features.npy")
fake_trans = np.load(feat_dir / "fake_transitions.npy")

fvd_uncond = frechet_video_distance(real_features, fake_features)
ds = diversity_score(fake_trans) if fake_trans.size else float("nan")

print(f"{'FVD (UNCONDITIONAL)':<22} {fvd_uncond:.4f}")
print(f"{'FVD (CONDITIONAL)':<22}  (not computed — see below)")
print(f"{'DIVERSITY SCORE':<22} {ds:.4f}")

FVD (UNCONDITIONAL)    321.0349
FVD (CONDITIONAL)       (not computed — see below)
DIVERSITY SCORE        4.7630
